# Aircraft Damage Exploration\n\nThis notebook is a lightweight companion to the refactored package. It keeps the project notebook-friendly while delegating the reusable logic to `src/aircraft_damage/`.\n

## Setup\n\nThe repository does not include the dataset or a trained classifier checkpoint. The cells below inspect the configured paths and only run inference if local assets are present.\n

In [ ]:
from pathlib import Path\nimport pprint\nimport sys\n\nPROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()\nSRC_ROOT = PROJECT_ROOT / 'src'\nif str(SRC_ROOT) not in sys.path:\n    sys.path.insert(0, str(SRC_ROOT))\n\nfrom aircraft_damage.config import load_config\nfrom aircraft_damage.predict import predict_image\nfrom aircraft_damage.report_generator import BLIPReportGenerator\n

In [ ]:
inference_config = load_config('configs/inference.yaml')\nreport_config = load_config('configs/report_generation.yaml')\npprint.pprint(inference_config)\npprint.pprint(report_config)\n

In [ ]:
checkpoint_path = Path(inference_config['paths']['model_checkpoint'])\ndataset_root = Path(inference_config['paths']['dataset_root'])\nprint('Checkpoint exists:', checkpoint_path.exists())\nprint('Dataset exists:', dataset_root.exists())\n

In [ ]:
# Replace with a real local image path once you have trained a model.\nsample_image = Path('path/to/local_aircraft_image.jpg')\n\nif checkpoint_path.exists() and sample_image.exists():\n    prediction = predict_image(\n        image_path=sample_image,\n        checkpoint_path=checkpoint_path,\n        class_names=inference_config['classifier']['class_names'],\n        image_size=tuple(inference_config['classifier']['image_size']),\n        threshold=float(inference_config['classifier']['threshold']),\n    )\n    print(prediction)\n\n    generator = BLIPReportGenerator(**report_config['report_generation'])\n    bundle = generator.generate_report_bundle(\n        image_path=sample_image,\n        predicted_class=prediction.predicted_class,\n        confidence=prediction.confidence,\n    )\n    print(bundle.report)\nelse:\n    print('Provide a trained checkpoint and a local image path to run the full pipeline from this notebook.')\n